Install libraries from requirements.txt

In [2]:
pip install -r requirements.txt --trusted-host pypi.org --trusted-host files.pythonhosted.org --trusted-host pypi.python.org


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from snowflake.snowpark import Session
import json
import os
import re
import io
import uuid

import pandas as pd
from openpyxl import load_workbook

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract
import pypdf
from PIL import Image, ImageFilter, ImageEnhance
from pdf2image import convert_from_path


In [4]:
BASE_DIR = os.getcwd()

excel_path      = os.path.join(BASE_DIR, "invoice_extract.xlsx")
input_directory = r'C:\Users\jakub.klosowski\OneDrive - Autoliv\Desktop\Invoices\Airspace'

# Tesseract executable path (Windows)
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Poppler path for PDF conversion (Windows)
POPPLER_PATH = r"C:\Program Files\poppler\poppler-25.12.0\Library\bin"

print("Excel Path:      ", excel_path)
print("Input Directory: ", input_directory)


Excel Path:       c:\Users\jakub.klosowski\email\FAP\AIRSPACE 3\invoice_extract.xlsx
Input Directory:  C:\Users\jakub.klosowski\OneDrive - Autoliv\Desktop\Invoices\Airspace


Update snowflake credentials

In [5]:
try:
    version = pytesseract.get_tesseract_version()
    print(f"Tesseract version: {version}")
except Exception as e:
    print(f"Tesseract not found or misconfigured: {e}")


Tesseract version: 5.5.0.20241111


In [6]:
connection_parameters = {
    "user":          "JAKUB.KLOSOWSKI@AUTOLIV.COM",
    "authenticator": "externalbrowser",
    "account":       "VN02639-YR60386",
    "warehouse":     "DEV_PRINCIPAL_WH",
    "database":      "PROTO_DEVTEAM_DB",
    "schema":        "PUBLIC",
}

session = Session.builder.configs(connection_parameters).create()
print("Snowflake session created.")


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/f3603cca-3b48-4625-afd2-2eb0587bf1d6/saml2?SAMLRequest=nZJBc9owEIX%2Fikc9WxYycYgGyFAYJsykhQKh096EvQYFWXIkGUN%2FfWUMM%2BkhOfTmkd%2Fu93bf9h9PhQyOYKzQaoA6mKAAVKozoXYD9LKehj0UWMdVxqVWMEBnsOhx2Le8kCUbVW6vlvBWgXWBb6Qsa34MUGUU09wKyxQvwDKXstXo2zOjmDBuLRjncehaklnhWXvnShZFdV3jOsba7CJKCInIQ%2BRVjeQLeocoP2eURjudankrOfmZPkB0ItJtEF7hCYtr4Veh2hV8Rtm2Isue1utFuJiv1igY3aYba2WrAswKzFGk8LJ8bg1Y7%2BDXMiFxL8G131sIldElYP6nMoCt0nUu%2BQFSXZSV892x%2F4pyyCKpd8LvbDYZoPIgssl2c96Vr9MpJMcfi%2FH8mK%2Fyn%2FPTJj9SedDGlq%2Brpzde78e0TlGwuSVMm4Rn1lYwU02uzj8RmoQkDilZk3tGY9ZNcO%2Be%2FEbBxPsTirtL5c38xQcuRGq01bnTSgoFrcvYj5WmPIy33V7YTehdyPOMhhS25K53v807WRI16VHUXhC7GDHD%2F91LP3rf5XqU331Os8lCS5Geg6k2BXcfx9jBncuLyML8ImVQcCFHWWbAWh%2BnlLoeG%2BDO374zFaBo2FL%2Fvf7hXw%3D%3D&RelayState=ver%3A3-hint%3A3302906936078-ETMsDgAAAZ0KIPs2ABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEMjXWIi1DpEk9cpltPPjgCsAAACQ%2

Test LLM Response

In [7]:
prompt = "What is your age?"

sql = f"""
    SELECT snowflake.cortex.complete(
        'llama3.1-70b',
        $$ {prompt.strip()} $$
    )
"""

# Execute the query and collect the result
result = session.sql(sql).collect()

# Print the LLM-generated response
print(result[0][0])

I was released to the p


In [8]:
def ask_cortex_invoice_data(session, extracted_text):
    try:
        prompt = f"""
Return only the JSON.

The following is an OCR extract of an Airspace Technologies invoice.
Your task is to extract the following fields with absolute precision.
Each field must be extracted exactly as it appears in the document.
If a field cannot be found, set its value to "N/A" (or [] for line_items). Do NOT copy values from other fields to fill missing ones.

The document layout:
- TOP: "Invoice" title, invoice number (XXXXXXX), invoice date
- LEFT BLOCK: "Bill To" — recipient address
- CENTER BLOCK: "Remit Payment To" — vendor/supplier details
- RIGHT BLOCK: TOTAL amount and Due Date
- ORDER TABLE: Terms | Due Date | Order Number | Order References | Commodity | Order Source | PO#
- TRANSPORT TABLE (columns left to right): Date | AWB | Tracking ID | Pickup Address | Delivery Address | Dims | POD
- ITEMS TABLE: Item | Description | Quantity | Amount (followed by Subtotal, Tax, Total, Amount Due)

Extract the following fields:

1. supplier
    The company issuing the invoice.
    Always set supplier to "Airspace Technologies, Inc". Do not extract from the document.

2. invoice_no
   The invoice number.
   Found at the top of the document directly under "Invoice", formatted as XXXXXXX.
   Extract without the # sign — e.g. "3003594".

3. invoice_date
   The invoice date.
   Found at the top, on the line directly below the invoice number.
   Format: M/D/YYYY — e.g. "9/3/2025".
   Important: All dates are written in the MM/DD/YYYY format (month/day/year). However, in the JSON response, always return dates in the DD/MM/YYYY format (day/month/year). For example, if the document shows "09/03/2025", return "03/09/2025" in the JSON.

4. bill_to
   The "Bill To" recipient name and address.
   Always set bill_to to 'Autoliv ASP, Inc. 3350 Airport Road Ogden, UT 84405   USA'. Do not extract from the document.

5. due_date
   The payment due date.
   Found on the line containing "Due Date:" in the header area (right block).
   Extract only the date value — e.g. "12/2/2025".
   If not found, set to "N/A".
   Important: All dates are written in the MM/DD/YYYY format (month/day/year). However, in the JSON response, always return dates in the DD/MM/YYYY format (day/month/year). For example, if the document shows "09/03/2025", return "03/09/2025" in the JSON.

6. order_number
   The sales order number.
   Found in the ORDER TABLE under the column "Order Number".
   Combine all rows — e.g. "Sales Order #3003594".

7. order_references
   The order reference value.
   Found in the ORDER TABLE under the column "Order References".
   Collect and join all rows — e.g. "PTN-947968" or "ITW Automotive GMBH A91855".

8. ship_date
   The shipment date.
   Found in the TRANSPORT TABLE under the column "Date" (leftmost column).
   e.g. "9/3/2025".
   Important: All dates are written in the MM/DD/YYYY format (month/day/year). However, in the JSON response, always return dates in the DD/MM/YYYY format (day/month/year). For example, if the document shows "09/03/2025", return "03/09/2025" in the JSON.

9. delivery_date
    The delivery date.
    Found in the TRANSPORT TABLE under the column "POD" (rightmost column).
    Extract the date from the first data row under the "POD" column.
    The date is always at the start of the text, in the format:

    Month (3-letter English abbreviation), Day (2 digits), Year (2 digits or 4 digits).
    Example: "Delivered at Oct 23 25 1450 PDT" → extract "Oct 23 25".
    Convert this to DD/MM/YYYY format, e.g. "Oct 23 25" → "23/10/2025".
    If not found, set to "N/A".

10. delivery time
    The delivery time.
    Found in the TRANSPORT TABLE under the column "POD" (rightmost column).
    Extract the time from the first data row under the "POD" column.
    The time is always at the end of the text, in the format:
    
    Hour (1 or 2 digits), Minute (2 digits), followed by "AM" or "PM".
    Example: "Delivered at Oct 23 25 1450 PDT" → extract "14:50".
    
    If not found, set to "N/A".

11. delivery_timezone
    The delivery time zone.
    Found in the TRANSPORT TABLE under the column "POD" (rightmost column), immediately after the time value.
    Extract the time zone abbreviation (e.g. "PDT", "CEST", "UTC").

    Example: "Delivered at Oct 23 25 1450 PDT" → extract "PDT".
    If not found, set to "N/A".

12. awb
    The Air Waybill number.
    Found in the TRANSPORT TABLE strictly under the column header "AWB" — the SECOND column.
    IMPORTANT: AWB and Tracking ID are completely separate columns. Do NOT copy the Tracking ID value into AWB.
    If the AWB column cell is blank or empty in the document, set awb to "N/A".
    e.g. "00610679126" or "N/A".

13. tracking_id
    The tracking identifier.
    Found in the TRANSPORT TABLE strictly under the column header "Tracking ID" — the THIRD column.
    This is always a short alphanumeric code, e.g. "ATEPWZ4NTT" or "ATK2ET2M9J".
    IMPORTANT: Do NOT copy this value into the awb field.

14. shipper
    The pickup / shipper address.
    Found in the TRANSPORT TABLE strictly under the column header "Pickup Address" — the FOURTH column.
    The Pickup Address column is positioned to the LEFT of the Delivery Address column.
    Collect ALL lines of text appearing under "Pickup Address" (can be multi-line company name + street + city + postal code).
    Join all lines with ", " — e.g. "NOVAPAX Kunststofftechnik Steiner GmbH & Co. KG, Schatzelbergstrasse 4-10, Berlin, 12099".
    If the Pickup Address column is blank, set to "N/A".

15. consignee
    The delivery / receiver address.
    Found in the TRANSPORT TABLE strictly under the column header "Delivery Address" — the FIFTH column.
    The Delivery Address column is positioned to the RIGHT of the Pickup Address column.
    Collect ALL lines of text appearing under "Delivery Address".
    Join all lines with ", " — e.g. "AMX, Av. de Los Sauces No. 9, Lerma, 52000".
    If the Delivery Address column is blank, set to "N/A".

16. weight_value
    The TOTAL shipment weight — sum of all weight values from ALL rows in the Dims column.
    The Dims column may contain multiple rows (separated by "; "), each ending with a weight, e.g.:
      "122.0*122.0*122.0cm, 95.0kg; 122.0*122.0*122.0cm, 95.0kg; 122.0*122.0*122.0cm, 95.0kg"
    Steps:
      1. Split dims by "; " to get individual rows.
      2. From each row, extract the last numeric value (the weight number before the unit).
      3. Sum all extracted weight numbers.
      4. Return the total as a numeric string — e.g. "285.0" or "98.0".
    If not found, set to "N/A".

17. weight_unit
    The unit of the shipment weight.
    Extract the unit attached to the weight numbers in the Dims column — e.g. "lbs" or "kg".
    All rows share the same unit; use the unit from the first row.
    If not found, set to "N/A".

18. line_items
    All charge lines from the ITEMS TABLE.
    Found after the header row containing "Item | Description | Quantity | Amount".
    For EACH data row (stop before Subtotal), extract:
      - item: first column (e.g. "Pickup")
      - description: second column (e.g. "SPECIAL RATE")
      - quantity: third column (numeric string, e.g. "0.0")
      - amount: fourth column — numeric value only, no $ sign — e.g. "2086.00"
    Return as a JSON array. If no rows, return [].

19. invoice_total
    The final invoice total.
    Found on the line labeled "Total" (not "Subtotal", not "Amount Due") in the totals block.
    Extract numeric value only, no $ sign — e.g. "2086.00".

20. invoice_currency
    The currency of the invoice.
    Determined from the $ symbol on the total or amount lines.
    Return 3-letter code — e.g. "USD".

=== RULES ===
- If a field is not found in the document, set its value to "N/A" (or [] for line_items).
- NEVER copy a value from one field into another to fill a blank — if the source column is empty, the field is "N/A".
- Do not infer or calculate values; extract only what is present in the OCR text.
- weight_value is the only exception — you must sum numeric weights from dims rows.
- Do not provide any explanation, only the JSON.
- Return a single JSON array with one object.

[
  {{
    "supplier": ...,
    "invoice_no": ...,
    "invoice_date": ...,
    "bill_to": ...,
    "due_date": ...,
    "order_number": ...,
    "order_references": ...,
    "ship_date": ...,
    "delivery_date": ...,
    "delivery_time": ...,
    "delivery_timezone": ...,
    "awb": ...,
    "tracking_id": ...,
    "shipper": ...,
    "consignee": ...,
    "dims": ...,
    "weight_value": ...,
    "weight_unit": ...,
    "line_items": [...],
    "invoice_total": ...,
    "invoice_currency": ...
  }}
]

Document text:
\"\"\"
{extracted_text}
\"\"\"
"""
        escaped_prompt = prompt.replace("'", "\\'")
        sql = f"""
            SELECT snowflake.cortex.complete(
                'llama3.1-70b',
                $$ {escaped_prompt} $$
            )
        """
        result = session.sql(sql).collect()
        response_text = result[0][0]
        return response_text.strip(), None
    except Exception as e:
        return None, f"❌ Failed to run Cortex Completion: {e}"


    Write data to excel

In [9]:
def insert_invoice_data(json_data, excel_path):
    """Appends a single invoice record to the Excel file (creates file if it doesn't exist).
    line_items (list of objects) is serialised to a JSON string in the LINE_ITEMS column.
    """
    if isinstance(json_data, str):
        json_data = json_data.strip("`\n ")
        json_data = json.loads(json_data)

    line_items = json_data.get("line_items", [])
    line_items_str = json.dumps(line_items, ensure_ascii=False) if line_items else ""

    row_data = {
        "SOURCE_FILE":       json_data.get("invoice_file", ""),
        "CARRIER_NAME_RAW":  json_data.get("supplier", ""),
        "INVOICE_ID":        json_data.get("invoice_no", ""),
        "INVOICE_DATE":      json_data.get("invoice_date", ""),
        "BILL_TO":           json_data.get("bill_to", ""),
        "DUE_DATE":          json_data.get("due_date", ""),
        "TERMS":             json_data.get("terms", ""),
        "ORDER_NUMBER":      json_data.get("order_number", ""),
        "ORDER_REFERENCES":  json_data.get("order_references", ""),
        "SHIP_DATE":         json_data.get("ship_date", ""),
        "AWB":               json_data.get("awb", ""),
        "TRACKING_ID":       json_data.get("tracking_id", ""),
        "SHIPPER":           json_data.get("shipper", ""),
        "CONSIGNEE":         json_data.get("consignee", ""),
        "DIMS":              json_data.get("dims", ""),
        "WEIGHT_VALUE":      json_data.get("weight_value", ""),
        "WEIGHT_UNIT":       json_data.get("weight_unit", ""),
        "LINE_ITEMS":        line_items_str,
        "INVOICE_TOTAL":     json_data.get("invoice_total", ""),
        "INVOICE_CURRENCY":  json_data.get("invoice_currency", ""),
    }
    df_new = pd.DataFrame([row_data])

    os.makedirs(os.path.dirname(excel_path), exist_ok=True)

    if os.path.exists(excel_path):
        book = load_workbook(excel_path)
        sheet = book["Sheet1"]
        startrow = sheet.max_row

        with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
            df_new.to_excel(writer, sheet_name="Sheet1", startrow=startrow, index=False, header=False)
    else:
        df_new.to_excel(excel_path, sheet_name="Sheet1", index=False)

    print(f"Inserted 1 row successfully into {excel_path}")



Image processing code to autorotate the images

In [10]:
def deskew_image_pil(pil_img, show=False):
    """
    Deskew image using horizontal projection profile.
    Scans angles from -15° to +15° and picks the one that maximises row-sum variance
    (well-aligned text produces sharp peaks in the row projection).
    """
    img = np.array(pil_img.convert("L"))

    # Binarise to isolate text pixels
    img_bin = cv2.adaptiveThreshold(
        img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 8
    )

    h, w = img_bin.shape
    center = (w // 2, h // 2)

    best_angle, best_score = 0.0, -1.0
    for angle in np.arange(-15, 15.1, 0.5):
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(
            img_bin, M, (w, h), flags=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0,
        )
        score = float(np.var(np.sum(rotated, axis=1)))
        if score > best_score:
            best_score, best_angle = score, angle

    print(f"Detected skew angle: {best_angle:.1f}°")

    M = cv2.getRotationMatrix2D(center, best_angle, 1.0)
    rotated_img = cv2.warpAffine(
        img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE
    )
    pil_rotated = Image.fromarray(rotated_img)

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        axes[0].imshow(pil_img, cmap="gray" if pil_img.mode == "L" else None)
        axes[0].set_title("Before deskew")
        axes[0].axis("off")
        axes[1].imshow(pil_rotated, cmap="gray")
        axes[1].set_title(f"After deskew ({best_angle:.1f}°)")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()

    return pil_rotated


In [11]:
def preprocess_image_pil(img):
    if img.mode != 'L':
        img = img.convert('L')

    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(1.2)


    enhancer_sharp = ImageEnhance.Sharpness(img)
    img = enhancer_sharp.enhance(1.2)

    img_np = np.array(img)
    img_blur = cv2.GaussianBlur(img_np, (1, 1), 0)
    _, img_otsu = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return Image.fromarray(img_otsu)

Image processing code to run tesseract OCR 

In [12]:
def _create_positioned_line(line_items, chars_per_pixel=17):
    """Reconstruct a single text line preserving original X-positions as spaces."""
    line_items.sort(key=lambda x: x["x"])
    if not line_items:
        return ""

    first = line_items[0]
    result = " " * max(0, first["x"] // chars_per_pixel) + first["text"]
    current_x = first["x"] + len(first["text"]) * chars_per_pixel

    for item in line_items[1:]:
        spaces = max(1, (item["x"] - current_x) // chars_per_pixel)
        result += " " * spaces + item["text"]
        current_x = item["x"] + len(item["text"]) * chars_per_pixel

    return result


def extract_with_real_positions(image_input, chars_per_pixel=17):
    """
    Run Tesseract OCR and reconstruct text with spatial layout preserved.
    Words are placed at their real pixel X positions so columns remain aligned.

    Args:
        image_input: file path (str) or PIL Image
        chars_per_pixel: pixel-to-character ratio controlling column spacing
    """
    image = Image.open(image_input) if isinstance(image_input, str) else image_input

    data = pytesseract.image_to_data(
        image,
        output_type=pytesseract.Output.DICT,
        config="--oem 1 --psm 4 --dpi 300",
    )

    # Keep all non-empty tokens (confidence filter disabled intentionally)
    tokens = [
        {"text": data["text"][i], "x": data["left"][i], "y": data["top"][i],
         "width": data["width"][i], "height": data["height"][i]}
        for i in range(len(data["text"]))
        if data["text"][i].strip()
    ]

    # Sort top-to-bottom, left-to-right and group into lines (±17 px tolerance)
    tokens.sort(key=lambda t: (t["y"], t["x"]))
    lines, current_line, current_y = [], [], None

    for token in tokens:
        if current_y is None or abs(token["y"] - current_y) < chars_per_pixel:
            current_line.append(token)
            current_y = token["y"]
        else:
            lines.append(_create_positioned_line(current_line, chars_per_pixel))
            current_line = [token]
            current_y = token["y"]

    if current_line:
        lines.append(_create_positioned_line(current_line, chars_per_pixel))

    return "\n".join(lines)


In [13]:
def _pdf_to_images(pdf_path, max_pages=None):
    """Convert PDF pages to PIL images using Poppler."""
    kwargs = {"dpi": 300, "poppler_path": POPPLER_PATH}
    if max_pages:
        kwargs.update({"first_page": 1, "last_page": max_pages})
    return convert_from_path(pdf_path, **kwargs)


def pdf_to_ocr_text(pdf_path, max_pages=None):
    """
    Convert all pages of a PDF to position-aware OCR text.
    Each page is wrapped with markers:
        === OCR: Full Page N (Real Positions) === ... === End of Page N ===
    """
    print(f"OCR: {os.path.basename(pdf_path)}"
          + (f" (first {max_pages} page(s))" if max_pages else ""))

    images = _pdf_to_images(pdf_path, max_pages)
    if not images:
        print("  ERROR: could not convert PDF to images.")
        return None

    pages = []
    for i, image in enumerate(images, 1):
        try:
            preprocessed = preprocess_image_pil(image)
        except Exception as exc:
            print(f"  WARN: preprocessing failed on page {i}: {exc} — using greyscale.")
            preprocessed = image.convert("L")

        page_text = extract_with_real_positions(preprocessed)
        pages.append(
            f"\n=== OCR: Full Page {i} (Real Positions) ===\n"
            f"{page_text.strip()}\n"
            f"=== End of Page {i} ==="
        )

    return "\n".join(pages).strip()


def get_numbered_ocr_text(pdf_path, max_pages=None):
    """Return OCR text with line numbers (useful for debugging LLM extraction)."""
    raw = pdf_to_ocr_text(pdf_path, max_pages)
    if not raw:
        return "ERROR: OCR failed."

    lines = raw.split("\n")
    numbered = [
        f"{i:3d}: {line}" if line.strip() else f"{i:3d}: [empty]"
        for i, line in enumerate(lines, 1)
    ]
    numbered.append(f"\nTotal lines: {len(lines)}")
    return "\n".join(numbered)


def batch_ocr_to_txt(input_dir, output_dir, max_pages=None):
    """Process all PDFs in input_dir, write one .txt OCR file per PDF to output_dir."""
    os.makedirs(output_dir, exist_ok=True)
    pdf_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".pdf")]
    if not pdf_files:
        print(f"No PDF files found in: {input_dir}")
        return

    for filename in pdf_files:
        txt_path = os.path.join(output_dir, os.path.splitext(filename)[0] + ".txt")
        text = get_numbered_ocr_text(os.path.join(input_dir, filename), max_pages)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)
        print(f"  Saved: {txt_path}")


In [14]:
batch_ocr_to_txt(input_directory, os.path.join(BASE_DIR, "ocr_output"), max_pages=1)

OCR: 118723.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\AIRSPACE 3\ocr_output\118723.txt
OCR: 3070034.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\AIRSPACE 3\ocr_output\3070034.txt
OCR: 3104466.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\AIRSPACE 3\ocr_output\3104466.txt
OCR: 3165491.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\AIRSPACE 3\ocr_output\3165491.txt
OCR: 3253904.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\AIRSPACE 3\ocr_output\3253904.txt
OCR: AIRSPACE  AIR Vendor Invoice Data Points (003).pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\AIRSPACE 3\ocr_output\AIRSPACE  AIR Vendor Invoice Data Points (003).txt


In [15]:

def process_directory_ocr_stage1(input_dir, max_pages=2):
    """
    Stage 1 pipeline: for each PDF in input_dir
      1. Run OCR  →  position-aware text
      2. Send text to Cortex LLM  →  structured JSON
      3. Append result to Excel
      4. Collect all records for Snowflake staging insert
    """
    print(f"Scanning: {input_dir}")
    print("=" * 60)

    raw_invoices_data = []
    local_excel_path  = os.path.join(input_dir, "invoice_extract.xlsx")
    total_saved       = 0

    for filename in sorted(os.listdir(input_dir)):
        if not filename.lower().endswith(".pdf"):
            continue

        print(f"\nProcessing: {filename}")
        extracted_text = pdf_to_ocr_text(
            os.path.join(input_dir, filename), max_pages=max_pages
        )

        if not extracted_text:
            print(f"  ERROR: OCR returned nothing for {filename}")
            continue

        structured_output, error = ask_cortex_invoice_data(session, extracted_text)
        if error:
            print(f"  ERROR (Cortex): {error}")
            continue

        try:
            invoices = json.loads(structured_output)
        except json.JSONDecodeError as e:
            print(f"  ERROR (JSON parse): {e}\n  Raw response: {structured_output}")
            continue

        for invoice in invoices:
            invoice["filename"] = filename
            try:
                # insert_invoice_data(invoice, local_excel_path)
                total_saved += 1
            except Exception as exc:
                print(f"  ERROR (Excel): {exc}")
        print("✅ Structured Data extracted:")
        print(json.dumps(invoices, indent=2, ensure_ascii=False))

        raw_invoices_data.extend(invoices)

    print("=" * 60)
    print(f"Invoices extracted : {len(raw_invoices_data)}")
    print(f"Saved to Excel     : {total_saved}")
    return raw_invoices_data


raw_invoices_data = process_directory_ocr_stage1(input_directory, max_pages=1)


Scanning: C:\Users\jakub.klosowski\OneDrive - Autoliv\Desktop\Invoices\Airspace

Processing: 118723.pdf
OCR: 118723.pdf (first 1 page(s))
✅ Structured Data extracted:
[
  {
    "supplier": "Airspace Technologies, Inc",
    "invoice_no": "3118723",
    "invoice_date": "23/10/2025",
    "bill_to": "Autoliv ASP, Inc. 3350 Airport Road Ogden, UT 84405   USA",
    "due_date": "21/0/2026",
    "order_number": "Sales Order #3118723",
    "order_references": "ITW Automotive GMBH A91855",
    "ship_date": "23/10/2025",
    "delivery_date": "23/10/2025",
    "delivery_time": "14:50",
    "delivery_timezone": "PDT",
    "awb": "02003286032",
    "tracking_id": "ATOFWENMHE",
    "shipper": "ITW Automotive GMBH, Bahnhofstrasse 50a, Hodenhagen, 29693",
    "consignee": "AUTOLIV AST DEPOT / CASAS INT, 9355 AIRWAY ROAD, SUITE 4, SAN DIEGO, CA, 92154",
    "weight_value": "335.0",
    "weight_unit": "kg",
    "line_items": [
      {
        "item": "Pickup",
        "description": "SPECIAL RATE",
     

In [16]:
import uuid
import json
import re


def _esc(val):
    if val is None or str(val).strip() in ('', 'N/A', 'n/a'):
        return 'NULL'
    return "'" + str(val).replace("'", "''") + "'"


def _parse_amount(s):
    try:
        return float(str(s).replace(',', ''))
    except Exception:
        return None


def _parse_weight_value(s):
    try:
        return float(str(s).strip().replace(',', '.'))
    except Exception:
        return None


def _build_order_references(order_references_raw):
    """
    Returns a list of (reference_type, reference_value) tuples for ORDER_REFERENCES.
    - If the value contains a PTN code (e.g. "PTN-947968"), extract it as type 'PTN'.
    - Otherwise insert the raw value as type 'ORDER_REFERENCES'.
    """
    if not order_references_raw or str(order_references_raw).strip() in ('', 'N/A', 'n/a'):
        return [('ORDER_REFERENCES', None)]

    raw = str(order_references_raw).strip()

    # Look for PTN pattern: PTN followed by - and alphanumeric, e.g. PTN-947968
    ptn_match = re.search(r'(PTN[-\s]?\w+)', raw, re.IGNORECASE)
    if ptn_match:
        ptn_value = ptn_match.group(1).strip()
        refs = [('PTN', ptn_value)]
        # If there's remaining text outside the PTN token, also store it as ORDER_REFERENCES
        remainder = raw.replace(ptn_match.group(0), '').strip().strip(',').strip()
        if remainder:
            refs.append(('ORDER_REFERENCES', remainder))
        return refs

    return [('ORDER_REFERENCES', raw)]


def insert_invoices_to_staging(
    session,
    raw_invoices_data,
    header_table='STG_INVOICE_HEADER',
    charge_table='STG_INVOICE_CHARGE_LINES',
    ref_table='STG_INVOICE_REFERENCES',
):
    if not raw_invoices_data:
        print("No data to insert.")
        return

    existing = {
        r[0] for r in session.sql(f"SELECT INVOICE_ID FROM {header_table}").collect() if r[0]
    }
    print(f"Already in {header_table}: {len(existing)} invoice(s)")

    inserted_h = inserted_c = inserted_r = 0

    for inv in raw_invoices_data:
        invoice_id = str(inv.get('invoice_no') or '').strip()
        if not invoice_id or invoice_id == 'N/A':
            print(f"  SKIP (no invoice_no): {inv.get('filename', '?')}")
            continue
        if invoice_id in existing:
            print(f"  SKIP (duplicate): {invoice_id}")
            continue

        weight_val  = _parse_weight_value(inv.get('weight_value'))
        weight_unit = inv.get('weight_unit')
        amount_val  = _parse_amount(inv.get('invoice_total'))
        currency    = inv.get('invoice_currency')

        # shipper może mieć literówkę "shipperr" — sprawdzamy oba klucze
        shipper_val = inv.get('shipper') or inv.get('shipperr')

        line_items = inv.get('line_items') or []
        if isinstance(line_items, str):
            try:
                line_items = json.loads(line_items)
            except Exception:
                line_items = []

        ocr_json    = json.dumps(inv, ensure_ascii=False)
        ocr_escaped = "'" + ocr_json.replace("'", "''") + "'"

        # STG_INVOICE_HEADER 
        session.sql(f"""
            INSERT INTO {header_table} (
                INVOICE_ID, INVOICE_DATE, DUE_DATE,
                SHIP_DATE, DELIVERY_DATE, DELIVERY_TIME, DELIVERY_TIMEZONE,
                CARRIER_NAME_RAW, BILL_TO_RAW,
                INVOICE_TOTAL, CURRENCY,
                TRANSPORT_MODE,
                SHIP_FROM, SHIP_TO,
                WEIGHT_VALUE, WEIGHT_UNIT,
                SOURCE_CARRIER, OCR_RAW_PAYLOAD,
                LOAD_TIMESTAMP, PROCESSING_STATUS
            )
            SELECT
                {_esc(invoice_id)},
                TRY_TO_DATE({_esc(inv.get('invoice_date'))}, 'DD/MM/YYYY'),
                TRY_TO_DATE({_esc(inv.get('due_date'))}, 'DD/MM/YYYY'),
                TRY_TO_DATE({_esc(inv.get('ship_date'))}, 'DD/MM/YYYY'),
                TRY_TO_DATE({_esc(inv.get('delivery_date'))}, 'DD/MM/YYYY'),
                {_esc(inv.get('delivery_time'))},
                {_esc(inv.get('delivery_timezone'))},
                {_esc(inv.get('supplier'))},
                {_esc(inv.get('bill_to'))},
                {amount_val if amount_val is not None else 'NULL'},
                {_esc(currency)},
                'Air',
                {_esc(shipper_val)},
                {_esc(inv.get('consignee'))},
                {weight_val if weight_val is not None else 'NULL'},
                {_esc(weight_unit)},
                {_esc(inv.get('supplier'))},
                PARSE_JSON({ocr_escaped}),
                CURRENT_TIMESTAMP(),
                'NEW'
        """).collect()
        inserted_h += 1

        # STG_INVOICE_CHARGE_LINES 
        for seq, item in enumerate(line_items, 1):
            if not isinstance(item, dict):
                continue
            charge_amount = _parse_amount(item.get('amount'))
            charge_qty    = _parse_amount(item.get('quantity'))
            session.sql(f"""
                INSERT INTO {charge_table} (
                    CHARGE_LINE_ID, INVOICE_ID, SOURCE_CARRIER,
                    CHARGE_SEQUENCE, CHARGE_DESCRIPTION_RAW,
                    CHARGE_QUANTITY, CHARGE_UNIT_PRICE,
                    CHARGE_AMOUNT, CURRENCY
                ) VALUES (
                    {_esc(str(uuid.uuid4()))},
                    {_esc(invoice_id)},
                    {_esc(inv.get('supplier'))},
                    {seq},
                    {_esc(item.get('description'))},
                    {charge_qty if charge_qty is not None else 'NULL'},
                    NULL,
                    {charge_amount if charge_amount is not None else 'NULL'},
                    {_esc(currency)}
                )
            """).collect()
            inserted_c += 1

        # ── STG_INVOICE_REFERENCES ────────────────────────────────────────
        # Fixed references: AWB and TRACKING_ID
        fixed_references = [
            ('AWB',         inv.get('awb')),
            ('TRACKING_ID', inv.get('tracking_id')),
        ]
        # Dynamic: PTN or ORDER_REFERENCES depending on value
        order_ref_entries = _build_order_references(inv.get('order_references'))

        for ref_type, ref_val in fixed_references + order_ref_entries:
            session.sql(f"""
                INSERT INTO {ref_table} (
                    REFERENCE_ID, INVOICE_ID, SOURCE_CARRIER,
                    REFERENCE_TYPE, REFERENCE_VALUE
                ) VALUES (
                    {_esc(str(uuid.uuid4()))},
                    {_esc(invoice_id)},
                    {_esc(inv.get('supplier'))},
                    {_esc(ref_type)},
                    {_esc(ref_val)}
                )
            """).collect()
            inserted_r += 1

        existing.add(invoice_id)
        print(f"  OK  {invoice_id}")

    print(f"\nSTG_INVOICE_HEADER:       {inserted_h} row(s)")
    print(f"STG_INVOICE_CHARGE_LINES: {inserted_c} row(s)")
    print(f"STG_INVOICE_REFERENCES:   {inserted_r} row(s)")


# Execute
session.sql("USE DATABASE DEV_SCM_DB").collect()
session.sql("USE SCHEMA OUT_TMS").collect()

insert_invoices_to_staging(session, raw_invoices_data)


Already in STG_INVOICE_HEADER: 20 invoice(s)
  SKIP (duplicate): 3118723
  SKIP (duplicate): 3070034
  SKIP (duplicate): 3104466
  SKIP (duplicate): 3165491
  SKIP (duplicate): 3253904
  SKIP (duplicate): 3003594

STG_INVOICE_HEADER:       0 row(s)
STG_INVOICE_CHARGE_LINES: 0 row(s)
STG_INVOICE_REFERENCES:   0 row(s)


In [17]:
def convert_mmddyyyy_to_ddmmyyyy(date_str):
    """Convert MM/DD/YYYY to DD/MM/YYYY. Returns original if not valid."""
    import re
    match = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{4})$', str(date_str).strip())
    if not match:
        return date_str
    mm, dd, yyyy = match.groups()
    return f"{dd.zfill(2)}/{mm.zfill(2)}/{yyyy}"

# Przykład: konwersja dat w rekordach
for invoice in raw_invoices_data:
    for key in ['invoice_date', 'due_date', 'ship_date', 'delivery_date']:
        if key in invoice:
            invoice[key] = convert_mmddyyyy_to_ddmmyyyy(invoice[key])